# External radiomics imputation controls

**Scientific purpose.** Inspect the five verified external W5 control conditions that
separate observed-feature and fold-median-imputation contributions.

**Runnability classification:** aggregate reanalysis is runnable from a clean clone.
Recreating patient-level control predictions requires private external features, internal
fold-training features, splits, and fitted W5 components and is intentionally not run here.

**Inputs:** released aggregate control table. **Outputs:** displayed aggregate comparison;
patient-level control predictions remain private.

No external labels were used to select an imputation condition. Because every external
case is known HCC, the reported measure is HCC sensitivity, not multiclass AUC.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd

CONTROL_DEFINITIONS = {
    "real_plus_imputed": "Observed external cells; missing cells use internal fold medians.",
    "median_only": "All selected cells use internal fold medians.",
    "strict_overlap_only": "Models restricted to features available in both domains.",
    "real_only_missing_zero": "Observed cells retained; missing cells set to zero.",
    "imputed_only_real_zero": "Observed cells zeroed; missing cells use internal fold medians.",
}
pd.DataFrame(
    [{"condition": name, "definition": definition} for name, definition in CONTROL_DEFINITIONS.items()]
)


## Deterministic matrix construction

The four direct cell-substitution conditions use fold-training medians only. The strict
overlap condition requires its separately verified fold-specific internal fitting path.


In [ ]:
def build_control_matrix(external_values, fold_medians, condition):
    values = np.asarray(external_values, dtype=np.float32)
    medians = np.asarray(fold_medians, dtype=np.float32).reshape(1, -1)
    if values.ndim != 2 or values.shape[1] != medians.shape[1]:
        raise ValueError("External values and fold medians are not feature-aligned.")
    observed = np.isfinite(values)
    if condition == "real_plus_imputed":
        return np.where(observed, values, medians)
    if condition == "median_only":
        return np.broadcast_to(medians, values.shape).copy()
    if condition == "real_only_missing_zero":
        return np.where(observed, values, 0.0)
    if condition == "imputed_only_real_zero":
        return np.where(observed, 0.0, medians)
    raise ValueError("Strict overlap uses its verified fold-specific model-fitting pathway.")


In [ ]:
aggregate_path = REPO_ROOT / "results" / "aggregate" / "radiomics_imputation_controls.csv"
if not aggregate_path.is_file():
    raise FileNotFoundError("Released aggregate radiomics control table is unavailable.")
controls = pd.read_csv(aggregate_path)
controls


## Verified aggregate findings

HCC sensitivity was 0.961 for real plus imputed, 1.000 for median-only, 0.243 for strict
overlap-only, 0.010 for real-only with missing values zeroed, and 0.971 for imputed-only
with observed values zeroed. Apparent external W5 performance was therefore strongly
influenced by the imputation pathway and is not evidence of robust transported radiomic
signal by itself.
